In [ ]:
# ── Cell 0: clone/update thesis repo from GitHub ────────────────────────────
import subprocess, os

REPO_URL = "https://github.com/richardtekere09/brain-tumor-mri-thesis.git"

if os.path.exists(".git"):
    # Already a git repo — just pull latest
    subprocess.run(["git", "pull"], check=True)
    print("Repo updated via git pull.")
elif os.path.exists("train.py"):
    # Files present but not a git repo — init and pull
    subprocess.run(["git", "init"], check=True)
    subprocess.run(["git", "remote", "add", "origin", REPO_URL], check=True)
    subprocess.run(["git", "fetch", "origin"], check=True)
    subprocess.run(["git", "checkout", "-f", "origin/main"], check=True)
    print("Repo initialised and checked out.")
else:
    # Empty or near-empty directory — clone into it
    subprocess.run(["git", "clone", REPO_URL, "repo_tmp"], check=True)
    import shutil
    for item in os.listdir("repo_tmp"):
        shutil.move(os.path.join("repo_tmp", item), item)
    shutil.rmtree("repo_tmp")
    print("Repo cloned.")


# Brain Tumor MRI — Training Notebook
**Thesis:** Нейросетевой анализ МРТ-изображений головного мозга для диагностики заболеваний мозга

Run this notebook on Kaggle GPU (P100/T4).  
Set `MODEL` to `model_a`, `model_b`, or `model_c` before running all cells.

In [ ]:
# ── Cell 1: choose model ─────────────────────────────────────────────────────
MODEL = 'model_a'   # change to 'model_b' or 'model_c'
print(f'Training: {MODEL}')

# ── Model C requirements ───────────────────────────────────────────────────
if MODEL == 'model_c':
    print()
    print('Model C selected. Additional requirements:')
    print('  1. Enable Internet in Kaggle Settings (BioBERT download ~440 MB)')
    print('  2. TextBraTS dataset must be attached (isaactekere/textbrats)')
    print('  3. First run downloads BioBERT — subsequent runs use HF cache')

In [ ]:
# ── Cell 2: install dependencies ─────────────────────────────────────────────
import subprocess, sys

packages = [
    'monai>=1.3',
    'transformers>=4.35',
    'accelerate',          # required for transformers model loading
    'nibabel>=5.0',
    'einops',
    'timm',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + packages, check=True)
print('Dependencies installed.')

In [ ]:
# ── Cell 3: mount data and set up paths ──────────────────────────────────────
import os, shutil, pathlib

# ── Kaggle dataset paths ───────────────────────────────────────────────────
BRATS_INPUT  = '/kaggle/input/datasets/awsaf49/brats20-dataset-training-validation'
USER_BASE    = '/kaggle/input/datasets/isaactekere'
SPLITS_INPUT = f'{USER_BASE}/brain-tumor-splits/brain-tumor-splits'
WEIGHTS_INPUT= f'{USER_BASE}/brain-tumor-weights/brain-tumor-weights'
# TextBraTS: TEXT_INPUT is the TextBraTSData folder itself.
# The symlink data/TextBraTS_hf must point to its PARENT so that
# data/TextBraTS_hf/TextBraTSData (= textbrats_root in config) resolves correctly.
TEXT_INPUT   = f'{USER_BASE}/textbrats/textbrats/TextBraTSData'
TEXT_PARENT  = os.path.dirname(TEXT_INPUT)   # .../textbrats/textbrats
# ──────────────────────────────────────────────────────────────────────────

os.makedirs('data', exist_ok=True)
os.makedirs('pretrained', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)

def make_symlink(link, candidates):
    """Create symlink to first existing candidate. Remove broken symlinks first."""
    if os.path.lexists(link):
        if os.path.exists(link):
            print(f'  Symlink OK: {link}')
            return
        else:
            os.unlink(link)  # broken symlink — remove and recreate
            print(f'  Removed broken symlink: {link}')
    for candidate in candidates:
        if os.path.exists(candidate):
            os.symlink(candidate, link)
            print(f'  Symlinked {link} -> {candidate}')
            return
    raise FileNotFoundError(f'None of these paths exist:\n' + '\n'.join(f'  {c}' for c in candidates))

print('Setting up BraTS data link...')
make_symlink('data/BraTS2020_TrainingData', [
    os.path.join(BRATS_INPUT, 'BraTS2020_TrainingData', 'MICCAI_BraTS2020_TrainingData'),
    os.path.join(BRATS_INPUT, 'MICCAI_BraTS2020_TrainingData'),
    os.path.join(BRATS_INPUT, 'BraTS2020_TrainingData'),
    BRATS_INPUT,
])

print('Setting up TextBraTS link...')
# Point to parent of TextBraTSData so config path "data/TextBraTS_hf/TextBraTSData" is valid
make_symlink('data/TextBraTS_hf', [
    TEXT_PARENT,   # preferred: parent dir → data/TextBraTS_hf/TextBraTSData works
    TEXT_INPUT,    # fallback: TextBraTSData itself
])

print('Copying splits.json...')
for candidate in [
    os.path.join(SPLITS_INPUT, 'splits.json'),
    os.path.join(USER_BASE, 'brain-tumor-splits', 'splits.json'),
]:
    if os.path.exists(candidate):
        shutil.copy2(candidate, 'data/splits.json')
        print(f'  Copied from {candidate}')
        break
else:
    raise FileNotFoundError(f'splits.json not found under {SPLITS_INPUT}')

print('Copying pretrained weights...')
for wf in ['resnet_18_23dataset.pth', 'swin_unetr_brats.pt']:
    dst = os.path.join('pretrained', wf)
    if os.path.exists(dst):
        print(f'  Already exists: {wf}')
        continue
    for candidate in [os.path.join(WEIGHTS_INPUT, wf), os.path.join(USER_BASE, 'brain-tumor-weights', wf)]:
        if os.path.exists(candidate):
            shutil.copy2(candidate, dst)
            print(f'  Copied {wf}')
            break
    else:
        print(f'  WARNING: {wf} not found')

n_cases = len(os.listdir('data/BraTS2020_TrainingData'))
print(f'\nData setup complete. BraTS cases: {n_cases}')

print('Setting up BioBERT local path...')
BIOBERT_PATH = None
import glob as _glob
# Search /kaggle/input for any folder containing config.json + pytorch_model.bin
for _cfg in _glob.glob('/kaggle/input/*/config.json') + _glob.glob('/kaggle/input/*/*/config.json'):
    _dir = os.path.dirname(_cfg)
    if os.path.exists(os.path.join(_dir, 'pytorch_model.bin')):
        BIOBERT_PATH = _dir
        print(f'  BioBERT found at: {BIOBERT_PATH}')
        break
if BIOBERT_PATH is None:
    print('  BioBERT dataset not attached — will download from HuggingFace (internet required)')


In [ ]:
# ── Cell 4: verify data accessible ───────────────────────────────────────────
import os, json

cases = sorted(os.listdir('data/BraTS2020_TrainingData'))
print(f'BraTS cases found: {len(cases)}')

with open('data/splits.json') as f:
    splits = json.load(f)
print(f'Splits — train: {len(splits["train"])} | val: {len(splits["val"])} | test: {len(splits["test"])}')

import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

In [ ]:
import os
# ── Patch model_c config to load BioBERT from local dataset ─────────────────
if MODEL == 'model_c' and BIOBERT_PATH:
    import yaml
    cfg_path = f'configs/{MODEL}.yaml'
    with open(cfg_path) as _f:
        _cfg = yaml.safe_load(_f)
    _cfg['model']['biobert_name'] = BIOBERT_PATH
    with open(cfg_path, 'w') as _f:
        yaml.dump(_cfg, _f, default_flow_style=False, allow_unicode=True)
    os.environ['TRANSFORMERS_OFFLINE'] = '1'
    print(f'Config patched: biobert_name -> {BIOBERT_PATH}')
    print('TRANSFORMERS_OFFLINE=1 (no HuggingFace requests)')
elif MODEL == 'model_c':
    print('No local BioBERT — downloading from HuggingFace (internet must be ON)')

# ── Cell 5: resume-aware training ────────────────────────────────────────────
import subprocess, sys

cmd = [
    sys.executable, 'train.py',
    '--config', f'configs/{MODEL}.yaml',
]
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, check=False)
print(f'Exit code: {result.returncode}')

In [ ]:
# ── Cell 6: verify checkpoints are ready for download ────────────────────────
import os

src_dir = f'checkpoints/{MODEL}'
print(f'Checkpoints in: /kaggle/working/{src_dir}')
print('Files:')
for fname in sorted(os.listdir(src_dir)):
    size_mb = os.path.getsize(os.path.join(src_dir, fname)) / 1024**2
    print(f'  {fname:<25} {size_mb:.1f} MB')
print('\nDownload from the Output tab on the right ->')


In [ ]:
# ── Cell 7: plot training curves ─────────────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt
import os

log_path = f'checkpoints/{MODEL}/train_log.csv'
if not os.path.exists(log_path):
    print('No train_log.csv found yet.')
else:
    df = pd.read_csv(log_path)
    print('Columns:', list(df.columns))

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle(f'{MODEL} -- Training Curves', fontsize=13, fontweight='bold')

    # ── Plot 1: Train loss vs Val loss ─────────────────────────────────────
    ax = axes[0]
    ax.plot(df['epoch'], df['train_loss'], label='Train loss', color='steelblue')
    if 'val_loss' in df.columns:
        val_loss_series = pd.to_numeric(df['val_loss'], errors='coerce')
        if val_loss_series.notna().any():
            ax.plot(df['epoch'], val_loss_series, label='Val loss', color='tomato')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Loss -- overfitting if val_loss >> train_loss')
    ax.legend()
    ax.grid(alpha=0.3)

    # ── Plot 2: Val Dice per region ────────────────────────────────────────
    ax = axes[1]
    for col, label, color in [
        ('val_dice_wt',   'WT',   '#4C72B0'),
        ('val_dice_tc',   'TC',   '#DD8452'),
        ('val_dice_et',   'ET',   '#C44E52'),
        ('val_dice_mean', 'Mean', '#55A868'),
    ]:
        if col in df.columns:
            ax.plot(df['epoch'], df[col], label=label, color=color)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Dice')
    ax.set_title('Validation Dice (higher = better)')
    ax.legend()
    ax.grid(alpha=0.3)

    # ── Plot 3: Learning rate schedule ────────────────────────────────────
    ax = axes[2]
    if 'lr' in df.columns:
        ax.plot(df['epoch'], df['lr'], color='purple')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Learning Rate')
        ax.set_title('LR Schedule')
        ax.set_yscale('log')
        ax.grid(alpha=0.3)

    plt.tight_layout()
    out_png = f'/kaggle/working/training_curves_{MODEL}.png'
    plt.savefig(out_png, dpi=130, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out_png}')

    # ── Summary ────────────────────────────────────────────────────────────
    dice_col = 'val_dice_mean'
    if dice_col in df.columns and df[dice_col].notna().any():
        best_idx = df[dice_col].idxmax()
        print(f'Best val Dice: {df[dice_col].max():.4f} at epoch {df.loc[best_idx, "epoch"]}')
        print(f'Final train loss: {df["train_loss"].iloc[-1]:.4f}')
        val_loss_num = pd.to_numeric(df["val_loss"], errors="coerce")
        if val_loss_num.notna().any():
            gap = val_loss_num.iloc[-1] - df["train_loss"].iloc[-1]
            print(f'Val loss gap: {gap:.4f}')
    if 'loss_seg' in df.columns:
        print(f'Final seg loss: {df["loss_seg"].iloc[-1]:.4f}')
    if 'loss_text' in df.columns:
        loss_text_num = pd.to_numeric(df["loss_text"], errors="coerce")
        if loss_text_num.notna().any():
            print(f'Final text loss: {loss_text_num.iloc[-1]:.4f}')
